# **FASE #1: Ingesta y Preprocesamiento Físico**

## **Módulo: Simulación Físico del Sensor HIKMICRO AD21P**

Esta función ha sido optimizada para resolver uno de los problemas más críticos en el procesamiento de audio para búsqueda y rescate: **el retardo de grupo y la distorsión de fase**. Al sustituir el filtro causal convencional (`lfilter`) por un filtrado de fase cero de doble paso (`filtfilt`), garantizamos que los frentes de onda y las componentes transitorias de los golpes rítmicos humanos (esenciales para discernir patrones de supervivencia en escombros) no sufran dispersión temporal. Esto previene que la red neuronal recurrente (RNN) reciba señales temporalmente desalineadas o deformadas.

Además, el bloque incorpora un generador estocástico de **Ruido Blanco Aditivo Gaussiano (AWGN)** para modelar con precisión el piso de ruido térmico y de ganancia instrumental real del preamplificador piezoeléctrico de contacto, evitando que el clasificador incurra en sobreajuste (*overfitting*) ante silencios matemáticos sintéticos.

In [1]:
import numpy as np
from scipy.signal import butter, filtfilt

def aplicar_simulador_ad21p(
    audio: np.ndarray,
    sr: int = 16000,
    lowcut: float = 800.0,
    highcut: float = 7500.0,
    noise_std: float = 1e-4,
) -> np.ndarray:
    """Aplica un filtro Butterworth pasa-banda de orden 4 de fase cero y añade ruido estático.

    Simula la respuesta en frecuencia física del sensor piezoeléctrico de contacto 
    HIKMICRO AD21P. El hardware real actúa como un filtro pasa-banda: corta los graves 
    por debajo de 800 Hz (aislando vibraciones mecánicas y sísmicas) y permite el paso 
    de frecuencias agudas hasta los 7500 Hz a través del medio físico.

    Para evitar artefactos artificiales de retraso de fase en los transitorios 
    de impacto de las víctimas, se ejecuta una operación de filtrado simétrico
    de fase cero (Forward-Backward Filtering) con filtfilt.

    Parameters
    ----------
    audio : np.ndarray
        Arreglo unidimensional (1D) de NumPy que contiene las muestras de audio.
    sr : int, optional
        Frecuencia de muestreo en Hz. Por defecto es 16000 Hz.
    lowcut : float, optional
        Frecuencia de corte inferior en Hz (Pasa-altos). Por defecto es 800.0 Hz.
    highcut : float, optional
        Frecuencia de corte superior en Hz (Pasa-bajos). Por defecto es 7500.0 Hz.
    noise_std : float, optional
        Desviación estándar para el piso de ruido electroacústico sutil (AWGN).
        Por defecto es 1e-4.

    Returns
    -------
    np.ndarray
        Señal de audio acondicionada espectralmente en float32.

    Raises
    ------
    TypeError
        Si 'audio' no es un np.ndarray o si 'sr' no se define como entero.
    ValueError
        Si el arreglo no es 1D, o si las frecuencias de corte rompen el límite de Nyquist.
    """
    # 1. Validaciones de Tipo y Estructura de Datos
    if not isinstance(audio, np.ndarray):
        raise TypeError("El parámetro 'audio' debe ser un arreglo de NumPy (np.ndarray).")

    if audio.ndim != 1:
        raise ValueError(f"El audio debe ser unidimensional (1D). Dimensiones recibidas: {audio.ndim}.")

    if not isinstance(sr, (int, np.integer)):
        raise TypeError(f"La frecuencia de muestreo 'sr' debe ser un número entero. Recibido: {type(sr)}.")

    # 2. Validación de Criterios Físicos de Nyquist
    nyquist = 0.5 * sr
    if lowcut <= 0 or highcut >= nyquist:
        raise ValueError(
            f"Los límites del filtro ({lowcut} Hz - {highcut} Hz) deben estar "
            f"estrictamente entre 0 y la frecuencia de Nyquist ({nyquist} Hz)."
        )
        
    if lowcut >= highcut:
        raise ValueError("La frecuencia 'lowcut' debe ser estrictamente menor que 'highcut'.")

    # Asegurar el cálculo de coeficientes de precisión con punto flotante doble (float64)
    audio_double = audio.astype(np.float64)

    # 3. Diseño del Filtro IIR Butterworth Pasa-Banda de Orden 4
    # Transformación a frecuencias normalizadas de Nyquist (rango 0.0 a 1.0)
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(N=4, Wn=[low, high], btype="bandpass", analog=False)

    # 4. Filtrado de Fase Cero (Forward-Backward Zero-Phase)
    audio_filtrado = filtfilt(b, a, audio_double)

    # 5. Modelado y Adición del Piso de Ruido Estático del Hardware (AWGN)
    piso_ruido = np.random.normal(loc=0.0, scale=noise_std, size=len(audio_filtrado))

    # 6. Fusión Física de Señales y Normalización a float32
    audio_final = (audio_filtrado + piso_ruido).astype(np.float32)

    return audio_final


### Justificación Técnica y Beneficios en la "Zona Cero"

1. **Retención de las Firmas Temporales (`filtfilt`):** En los filtros causales estándar (`lfilter`), las frecuencias cercanas a la frecuencia de corte sufren retrasos de tiempo variables (retardo de grupo). Esto desfigura el "silencio relativo" entre golpes consecutivos y altera las envolventes de decaimiento natural de la onda acústica en el escombro. Al procesar el audio bidireccionalmente, `filtfilt` hace que la fase sea exactamente cero en todas las frecuencias, preservando la nitidez de los impactos del Código Morse o golpes rítmicos (`Target_Rhythmic`).
2. **Robustez ante el Silencio Absoluto:** Los conjuntos de datos como *Common Voice* o grabaciones sintéticas aisladas suelen tener tramos de silencio absoluto (amplitud cero pura). Si la red se entrena con ceros puros en sus zonas silenciosas, fallará catastróficamente al desplegarse en el terreno con el sensor físico, ya que interpretará el leve siseo del canal analógico del AD21P como una señal predictiva. El piso de ruido simulado con `noise_std = 1e-4` inmuniza al clasificador Wide & Deep frente a estas variaciones operativas.
3. **Optimización de Cómputo (Precision float32):** El algoritmo ejecuta internamente los cálculos de filtrado en punto flotante de doble precisión (`float64`) para evitar errores acumulativos de redondeo digital en los coeficientes IIR. No obstante, la salida es casteada a precisión simple (`float32`), la cual es el estándar nativo de aceleración matemática para PyTorch/TensorFlow en la GPU RTX 3060 de campo.

## **Módulo: Emulación de Ruido Estático (AWGN)**

Este bloque está diseñado bajo rigurosos criterios de la teoría de Procesamiento Digital de Señales (PDS) y optimizado para actuar como un módulo robusto de aumento de datos (*data augmentation*) dentro de la **Fase 1** de pipeline USAR.

In [2]:
import numpy as np


def inyectar_ruido_estatico(
    audio: np.ndarray,
    snr_db_min: float = 15.0,
    snr_db_max: float = 40.0
) -> np.ndarray:
    """Inyecta ruido blanco gaussiano aditivo (AWGN) con relación señal-ruido dinámica.

    Simula el piso de ruido térmico, de ganancia y de cuantificación analógico-digital
    intrínseco de los transductores piezoeléctricos de contacto, como el HIKMICRO AD21P.
    La inyección estocástica de ruido previene que la red neuronal (RNN Wide & Deep)
    sufra de un choque de distribución (domain shift) al procesar grabaciones reales
    de campo con estática, inmunizándola contra silencios digitales artificiales.

    Parameters
    ----------
    audio : np.ndarray
        Arreglo unidimensional (1D) de NumPy que contiene las muestras de audio.
    snr_db_min : float, optional
        Límite inferior para la selección aleatoria de la relación señal-ruido (SNR)
        en decibelios (dB). Por defecto es 15.0 dB (señal altamente ruidosa).
    snr_db_max : float, optional
        Límite superior para la selección aleatoria de la relación señal-ruido (SNR)
        en decibelios (dB). Por defecto es 40.0 dB (señal con ruido muy sutil).

    Returns
    -------
    np.ndarray
        Señal de audio acondicionada con ruido aditivo, normalizada y casteada 
        a precisión de punto flotante simple (float32) para uso directo en GPU.

    Raises
    ------
    TypeError
        Si 'audio' no es un arreglo np.ndarray.
    ValueError
        Si 'audio' no es unidimensional (1D), o si el rango de SNR definido 
        es numéricamente inconsistente (snr_db_min >= snr_db_max).
    """
    # 1. Validaciones estrictas de tipo y dimensiones de entrada
    if not isinstance(audio, np.ndarray):
        raise TypeError(
            "El parámetro 'audio' debe ser un arreglo de NumPy (np.ndarray)."
        )

    if audio.ndim != 1:
        raise ValueError(
            f"El audio debe ser unidimensional (1D). Dimensiones recibidas: {audio.ndim}."
        )

    if snr_db_min >= snr_db_max:
        raise ValueError(
            f"El límite inferior de SNR ({snr_db_min} dB) debe ser "
            f"estrictamente menor que el límite superior ({snr_db_max} dB)."
        )

    # 2. Selección estocástica del SNR para generalización del modelo (Regularización)
    snr_db = np.random.uniform(snr_db_min, snr_db_max)

    # 3. Cálculo de la potencia cuadrática media (RMS^2) de la señal de entrada
    # Se realiza el cálculo intermedio en float64 para mitigar errores de redondeo
    audio_double = audio.astype(np.float64)
    potencia_senal = np.mean(audio_double ** 2)

    # 4. SALVAGUARDA MATEMÁTICA CONTRA SILENCIO ABSOLUTO (Evitar indeterminación o NaN)
    # En audios sintéticos con tramos de silencio absoluto digital, la potencia es 0.0.
    # Un valor de 0.0 provocaría una potencia de ruido nula (sigma_ruido = 0.0) o divisiones
    # por cero al normalizar. Aplicamos un piso de potencia de seguridad a -100 dB FS.
    piso_potencia_seguridad = 1e-10
    if potencia_senal < piso_potencia_seguridad:
        potencia_senal = piso_potencia_seguridad

    # 5. Obtención de la potencia y desviación estándar del ruido requeridas
    # Derivado de: SNR_dB = 10 * log10(P_signal / P_noise)
    potencia_ruido = potencia_senal / (10 ** (snr_db / 10.0))
    sigma_ruido = np.sqrt(potencia_ruido)

    # 6. Generación del ruido blanco gaussiano aditivo (AWGN) de media cero
    # Cumple con la propiedad estadística de media cero (me = 0) y densidad espectral
    # de potencia constante (Pee = sigma^2) en todo el ancho de banda.
    ruido_gaussiano = np.random.normal(
        loc=0.0, scale=sigma_ruido, size=len(audio)
    )

    # 7. Fusión aditiva y casteo a float32 para optimización del DataLoader en GPU
    audio_ruidoso = (audio_double + ruido_gaussiano).astype(np.float32)

    return audio_ruidoso


### **Análisis Físico y Matemático del Bloque**

1. **Modelado Estadístico de Ruido Blanco Gaussiano (AWGN):**  
   Matemáticamente, definimos el ruido estático como una secuencia aleatoria \\(e[n]\\) estacionaria en sentido amplio, con media cero (\\(\mu_e = 0\\)) y una función de autocorrelación \\(\phi_{ee}[m] = \sigma_e^2 \delta[m]\\). Esto garantiza que su densidad espectral de potencia sea plana y constante para todas las frecuencias del espectro (\\(\Phi_{ee}(e^{j\omega}) = \sigma_e^2\\)). En el dominio físico del sensor HIKMICRO AD21P, esto emula perfectamente la superposición del ruido térmico Johnson-Nyquist y el ruido de cuantificación uniforme del convertidor analógico-digital (A/D).

2. **La Salvaguarda del Silencio Absoluto:**  
   En un pipeline de producción USAR, es común recibir fragmentos sintéticos de *Common Voice* o del generador de Morse que contienen tramos de silencio digital absoluto (amplitud pura de cero) antes de la oclusión o centralización. Si calculamos la potencia de señal sobre un bloque de ceros, obtenemos \\(P_{signal} = 0.0\\). Sin salvaguarda, la ecuación:
   \\[P_{noise} = \frac{0.0}{10^{\frac{SNR}{10}}} = 0.0 \implies \sigma_e = 0.0\\]
   generaría un vector de ruido con ceros absolutos, manteniendo el silencio plano intacto. Al implementar `piso_potencia_seguridad = 1e-10` (equivalente a \\(-100 \text{ dB FS}\\)), garantizamos que incluso en ausencia de señal útil se inyecte un piso de ruido electroacústico sutil pero existente, protegiendo al modelo de aprender que el silencio digital absoluto es una característica válida de entrenamiento.

3. **Coherencia con las Regras de Normalización de la Red Neural:**  
   La normalización de los datos de entrada es un paso crítico en el Deep Learning. Si las amplitudes acústicas varían de forma brusca e impredecible entre muestras limpias y ruidosas, los gradientes del optimizador Adam en la red Wide & Deep fluctuarán de forma inestable, ralentizando o impidiendo la convergencia del entrenamiento (efecto *zigzag*). Al inyectar un ruido dinámico que se auto-escala proporcionalmente a la potencia de la señal útil (\\(P_{signal}\\)), preservamos la consistencia energética del espectrograma y de las heurísticas globales (ZCR, RMS) de la rama Wide, permitiendo que el optimizador descienda con un gradiente más suave y redondo hacia el óptimo global.

## **Módulo: Padding por crosfade**

La función **`padding_por_crossfade`** resolve el problema de las costuras acústicas mediante una igualación de potencia RMS y un fundido cruzado (*crossfade*) de 50 ms.


In [3]:
import numpy as np

def padding_por_crossfade(
    senal_util: np.ndarray,
    ruido_ambiental: np.ndarray,
    target_duration: float = 4.0,
    sr: int = 16000
) -> np.ndarray:
    """Estandariza audios cortos aplicando un lienzo de ruido con crossfading y RMS matching.

    Parameters
    ----------
    senal_util : np.ndarray
        Señal útil de entrada (voz o impacto rítmico).
    ruido_ambiental : np.ndarray
        Pista de ruido de fondo para rellenar los extremos.
    target_duration : float, optional
        Duración de salida en segundos. Por defecto es 4.0 s (64,000 muestras).
    sr : int, optional
        Frecuencia de muestreo. Por defecto es 16000 Hz.

    Returns
    -------
    np.ndarray
        Audio de salida continuo de tamaño exacto (target_duration * sr) en float32.
    """
    total_samples = int(target_duration * sr)
    len_util = len(senal_util)

    # 1. Truncado inmediato si excede la duración objetivo
    if len_util >= total_samples:
        return senal_util[:total_samples].astype(np.float32)

    # Procesar en float64 para evitar inestabilidad en operaciones matemáticas
    senal_util_64 = senal_util.astype(np.float64)
    ruido_ambiental_64 = ruido_ambiental.astype(np.float64)

    # 2. Lienzo de Fondo
    if len(ruido_ambiental_64) > total_samples:
        start_rand = np.random.randint(0, len(ruido_ambiental_64) - total_samples)
        canvas = ruido_ambiental_64[start_rand : start_rand + total_samples].copy()
    else:
        canvas = np.resize(ruido_ambiental_64, total_samples).copy()

    # 3. RMS Matching (Igualación de Energía)
    # Evaluamos los primeros y últimos 50 ms de la señal útil (ruido ya existente en el clip)
    edge_len = int(0.05 * sr)  # 50 ms
    if len_util >= 2 * edge_len:
        edge_samples = np.concatenate([senal_util_64[:edge_len], senal_util_64[-edge_len:]])
    else:
        edge_samples = senal_util_64.copy()

    rms_edges = np.sqrt(np.mean(edge_samples ** 2) + 1e-10)
    rms_canvas = np.sqrt(np.mean(canvas ** 2) + 1e-10)
    rms_global = np.sqrt(np.mean(senal_util_64 ** 2) + 1e-10)

    # Si el audio fuente tiene compuerta de ruido (silencio absoluto en los bordes),
    # anclamos el volumen del ruido AD21P a -15 dB (0.177 lineal) del RMS global de la señal útil.
    if rms_edges < 1e-4:
        rms_edges = rms_global * 0.177

    # Escalar el fondo para que coincida de forma segura con la energía
    canvas_scaled = canvas * (rms_edges / rms_canvas)

    

    # 4. Centralización e Inserción con Crossfading en costuras
    start_idx = (total_samples - len_util) // 2
    end_idx = start_idx + len_util

    y = canvas_scaled.copy()

    # Asegurar que el crossfade no sea mayor que la mitad de la señal útil
    fade_len = int(0.05 * sr)
    if len_util < 2 * fade_len:
        fade_len = len_util // 2

    if fade_len > 0:
        # Costura Izquierda: Ruido disminuye (1 - alpha), Señal útil aumenta (alpha)
        alpha = np.linspace(0.0, 1.0, fade_len)
        y[start_idx : start_idx + fade_len] = (
            (1.0 - alpha) * canvas_scaled[start_idx : start_idx + fade_len] +
            alpha * senal_util_64[:fade_len]
        )

        # Costura Derecha: Señal útil disminuye (beta), Ruido aumenta (1 - beta)
        beta = np.linspace(1.0, 0.0, fade_len)
        y[end_idx - fade_len : end_idx] = (
            beta * senal_util_64[-fade_len:] +
            (1.0 - beta) * canvas_scaled[end_idx - fade_len : end_idx]
        )

        # Porción Central: Reemplazo puro (mantiene la señal intacta)
        if len_util > 2 * fade_len:
            y[start_idx + fade_len : end_idx - fade_len] = senal_util_64[fade_len : len_util - fade_len]
    else:
        y[start_idx : end_idx] = senal_util_64

    return y.astype(np.float32)


### Aspectos Clave de la Solución
* **Fase y Amplitud Continuas:** Al calcular la potencia RMS de las colas de 50 ms (`rms_edges`), el siseo del lienzo se unifica matemáticamente con el siseo que arrastra el clip de audio, eliminando los saltos abruptos de energía en la costura.
* **Preservación Central:** Las transiciones de desvanecimiento con `np.linspace` operan únicamente en las costuras (`fade_len`). La información central del audio útil (la firma vocal o el impacto transitorio) no se distorsiona con sumas adicionales de ruido.
* **Seguridad Numérica:** Se incorpora un épsilon de `1e-10` para prevenir divisiones por cero (`NaN`) en silencios absolutos.

# Modulo: Aumento de variabilidad para el dataset morse

Para evitar que el modelo se confunda y obligarlo a prestar atención solo a la cadencia rítmica y no al material, debemos aplicar una técnica de Data Augmentation en el dominio del tiempo y la frecuencia antes de hacer el padding:

Pitch Shifting (Cambio de Tono): Sube o baja artificialmente las frecuencias del audio. Esto engaña a la red simulando que el mismo ritmo fue golpeado en diferentes materiales (piedra, madera, metal grueso, metal delgado).

Time Stretching (Estiramiento Temporal): Acelera o ralentiza el audio. Esto simula el estado físico del sobreviviente (una cadencia rápida si está en pánico, o una cadencia muy lenta si está exhausto).

In [4]:
import numpy as np
import librosa

def simular_variabilidad_impacto(
    audio: np.ndarray, 
    sr: int = 16000, 
    variar_cadencia: bool = True, 
    variar_material: bool = True
) -> np.ndarray:
    """Atera la física del sonido para generalizar el patrón rítmico.
    
    Aplica Time-Stretching para simular estados de fatiga o pánico en el
    sobreviviente, y Pitch-Shifting para simular diferentes materiales 
    de impacto (concreto, metal, piedra, madera).
    
    Parameters
    ----------
    audio : np.ndarray
        Señal acústica útil de entrada (secuencia rítmica).
    sr : int, optional
        Frecuencia de muestreo. Por defecto es 16000 Hz.
    variar_cadencia : bool, optional
        Si es True, altera la velocidad del ritmo entre 0.8x y 1.2x.
    variar_material : bool, optional
        Si es True, altera las resonancias entre -4 y +4 semitonos.
        
    Returns
    -------
    np.ndarray
        Señal acústica con la variabilidad aplicada en float32.
    """
    audio_modificado = audio.copy()
    
    # 1. Variabilidad de Cadencia (Sobreviviente exhausto vs en pánico)
    if variar_cadencia:
        # rate < 1.0 (más lento/exhausto), rate > 1.0 (más rápido/pánico)
        rate = np.random.uniform(0.3, 1.5)
        audio_modificado = librosa.effects.time_stretch(y=audio_modificado, rate=rate)
        
    # 2. Variabilidad de Material (Simulación de densidad de escombros)
    if variar_material:
        # n_steps < 0 (sonido sordo/roca), n_steps > 0 (sonido metálico/agudo)
        n_steps = np.random.uniform(-6.0, 6.0)
        audio_modificado = librosa.effects.pitch_shift(y=audio_modificado, sr=sr, n_steps=n_steps)
        
    return audio_modificado.astype(np.float32)

##  **Módulo: Enrutador es_dataset_morse**

Para enrutar correctamente los audios dentro del Orquestador, crearemos una función de validación rápida. Puedes colocarla en la misma celda donde tengas es_dataset_zona_cero.

In [5]:
def es_dataset_morse(dataset_origen: str) -> bool:
    """Verifica si el dataset de origen corresponde a la categoría de impactos rítmicos.
    
    Identifica si la carpeta de origen es el dataset masivo de Código Morse para
    habilitar la inyección de Data Augmentation físico.
    """
    nombres_validos = ["morse", "codigo_morse", "dataset_morse"]
    return dataset_origen.strip().lower() in nombres_validos

## **Módulo: Hacer recortes de 4s**

El módulo **`segmentar_y_estandarizar_audio`** esta optimizado mediante slicing vectorial en NumPy. Para cumplir con el límite de procesamiento y la restricción de longitud, el código es compacto, robusto y está diseñado para evitar bucles `for` costosos en la segmentación principal.

In [6]:

import numpy as np
from typing import List

def segmentar_y_estandarizar_audio(
    senal_util: np.ndarray,
    ruido_ambiental: np.ndarray,
    sr: int = 16000,
    target_duration: float = 4.0,
    min_duration_residual: float = 1.0
) -> List[np.ndarray]:
    """Segmenta un audio largo en ventanas contiguas de 4s y estandariza colas con padding por crossfade.

    Parameters
    ----------
    senal_util : np.ndarray
        Señal acústica útil de entrada.
    ruido_ambiental : np.ndarray
        Pista de ruido de fondo para rellenar los extremos del residuo.
    sr : int, optional
        Frecuencia de muestreo en Hz. Por defecto es 16000 Hz.
    target_duration : float, optional
        Duración de la ventana de salida en segundos. Por defecto es 4.0 s.
    min_duration_residual : float, optional
        Duración mínima del residuo en segundos para no ser descartado. Por defecto es 1.0 s.

    Returns
    -------
    List[np.ndarray]
        Lista de arreglos NumPy de longitud exacta de muestras (64,000 para 4.0s a 16 kHz).
    """
    try:
        # Validaciones de integridad física de los datos
        if not isinstance(senal_util, np.ndarray) or not isinstance(ruido_ambiental, np.ndarray):
            raise TypeError("Los parámetros de entrada deben ser arreglos NumPy (np.ndarray).")
        if senal_util.ndim != 1 or ruido_ambiental.ndim != 1:
            raise ValueError("Las señales acústicas deben ser estrictamente unidimensionales (1D).")

        window_size = int(target_duration * sr)
        min_residual_size = int(min_duration_residual * sr)
        total_samples = len(senal_util)

        # Caso 1: El audio es más corto que la ventana objetivo
        if total_samples < window_size:
            if total_samples >= min_residual_size:
                return [padding_por_crossfade(senal_util, ruido_ambiental, target_duration, sr)]
            return []

        # Caso 2: Segmentación contigua utilizando slicing matricial de NumPy (Evita bucles lentos)
        n_windows = total_samples // window_size
        sliced_signal = senal_util[:n_windows * window_size]
        
        # Reshaping directo para crear la matriz de ventanas
        grid_windows = sliced_signal.reshape(n_windows, window_size)
        segmentos = [grid_windows[i].copy().astype(np.float32) for i in range(n_windows)]

        # Procesamiento condicional del residuo final (cola)
        residual_start = n_windows * window_size
        residual = senal_util[residual_start:]
        
        if len(residual) >= min_residual_size:
            padded_residual = padding_por_crossfade(residual, ruido_ambiental, target_duration, sr)
            segmentos.append(padded_residual)

        return segmentos

    except Exception as e:
        print(f"⚠️ [AVISO TÁCTICO PDS] Fallo en segmentación secuencial: {e}")
        return []


### Detalles de Optimización
* **Evita Bucles `for` en Segmentación:** Al utilizar `.reshape(n_windows, window_size)` nativo de NumPy, la división del audio principal se realiza en microsegundos directamente en memoria C, liberando de carga a la CPU.
* **Integración del Residuo:** Si el fragmento final tiene una longitud válida (\\(\ge 1.0\\) segundo), se asila y se manda a `padding_por_crossfade` para su estandarización a 4 segundos exactos (64,000 muestras). Si mide menos, se descarta para evitar falsas detecciones por falta de información espectral.

## **Módulo: Lógica Delta (Caché)**

El **Módulo de Lógica Delta (Caché)** diseñado específicamente para optimizar los tiempos de preprocesamiento de la Fase 1 del pipeline USAR:

In [7]:
import os
from pathlib import Path
from typing import List, Union

def es_audio_nuevo(ruta_input: Union[str, Path], dir_output: Union[str, Path]) -> bool:
    """Verifica si un audio de entrada es nuevo o ya ha sido procesado (Lógica Delta).

    Comprueba si en el directorio de salida ya existen archivos procesados cuyo
    nombre inicie con el nombre base del archivo de entrada, soportando así la
    segmentación de archivos (ej. 'audio_01_part1.wav').

    Parameters
    ----------
    ruta_input : Union[str, Path]
        Ruta del archivo de audio de entrada (.wav).
    dir_output : Union[str, Path]
        Directorio donde se almacenan los tensores o audios preprocesados.

    Returns
    -------
    bool
        True si el archivo es nuevo (procesar), False si ya existe (omitir).
    """
    try:
        path_input = Path(ruta_input)
        path_output_dir = Path(dir_output)
        
        base_name = path_input.stem # Ej: "audio_01"
        
        if not path_output_dir.exists():
            return True
            
        # Escaneo eficiente mediante iteradores de Pathlib
        for item in path_output_dir.iterdir():
            if item.is_file() and item.name.startswith(base_name):
                return False  # Se detecta coincidencia (omitir)
                
        return True
    except Exception as e:
        print(f"⚠️ [AVISO] Error al verificar Lógica Delta para {ruta_input}: {e}")
        return True  # Por seguridad en rescate, se procesa ante fallas

def obtener_audios_pendientes(input_dir: Union[str, Path], output_dir: Union[str, Path]) -> List[str]:
    """Filtra y retorna exclusivamente los archivos .wav que requieren procesamiento.

    Parameters
    ----------
    input_dir : Union[str, Path]
        Directorio con los audios crudos capturados por el sensor AD21P.
    output_dir : Union[str, Path]
        Directorio de destino de preprocesamiento.

    Returns
    -------
    List[str]
        Lista de rutas absolutas a archivos pendientes. Retorna lista vacía si
        no hay archivos nuevos.

    Raises
    ------
    FileNotFoundError
        Si el directorio de entrada 'input_dir' no existe.
    """
    path_input = Path(input_dir)
    path_output = Path(output_dir)
    
    if not path_input.exists():
        raise FileNotFoundError(f"Directorio de entrada inexistente: {input_dir}")
        
    # Creación automática y robusta del directorio de destino si no existe
    if not path_output.exists():
        path_output.mkdir(parents=True, exist_ok=True)
        
    pendientes = []
    try:
        for file_path in path_input.iterdir():
            if file_path.is_file() and file_path.suffix.lower() in [".wav", ".mp3"]:
                abs_path = str(file_path.resolve())
                if es_audio_nuevo(abs_path, output_dir):
                    pendientes.append(abs_path)
        return pendientes
    except Exception as e:
        print(f"❌ [ERROR] Fallo al escanear audios pendientes: {e}")
        return []

def hay_audios_sin_preprocesar(input_dir: Union[str, Path]) -> bool:
    """Rombo de control: ¿Existen archivos .wav/.mp3 crudos por evaluar?"""
    path_input = Path(input_dir)
    if not path_input.exists():
        return False
    # Retorna True si encuentra al menos un archivo .wav
    return any(f.is_file() and f.suffix.lower() in [".wav", ".mp3"] for f in path_input.iterdir())


### Beneficios de la Solución
* **Omitir Computación Repetitiva:** Si el algoritmo se interrumpe o se agregan nuevos archivos en caliente, la función evita reprocesar horas de audio que ya fueron filtradas, centralizadas o segmentadas en pasadas ejecuciones.
* **Alineación con el Diagrama de Flujo:** Sigue fielmente la compuerta condicional del bloque de decisión **`¿El audio es nuevo? -> OMITE`** descrita en tu arquitectura técnica.

##  **Módulo: Enrutador es_dataset_zona_cero**

La función lógica de control de flujo **`es_dataset_zona_cero`** actúa como el **Enrutador** garantizando que el audio capturado directamente por el sensor físico HIKMICRO AD21p en terreno (*Zona Cero*) no sufra un doble filtrado ni una inyección redundante de estática 

In [8]:
def es_dataset_zona_cero(dataset_origen: str) -> bool:
    """Determina si el dataset de origen corresponde a datos reales de campo.

    Esta función de control de flujo implementa el bloque condicional lógico 
    "¿El audio es de Dataset Zona Cero?" en la Fase 1 del pipeline USAR V2.0. 
    Permite decidir de manera determinista si un archivo requiere emulación 
    física de canal (atenuación por concreto y siseo de hardware) o si puede 
    pasar directamente al bloque de Estandarización Temporal.

    Parameters
    ----------
    dataset_origen : str
        Identificador del dataset de origen. Los valores estándar del pipeline son:
        - "zona_cero" : Grabaciones reales de terreno (HIKMICRO AD21P).
        - "morse" : Dataset sintético de impactos rítmicos en Morse.
        - "voces" : Grabaciones del dataset Common Voice.
        - "esc_50" : Base de datos de ruidos y transitorios ambientales.
        - "urbansound" : Base de datos de maquinaria de rescate.

    Returns
    -------
    bool
        True si el origen es "zona_cero" (indica omitir simulación física),
        False en caso contrario (indica enrutar a simulación de oclusión y ruido).

    Raises
    ------
    TypeError
        Si 'dataset_origen' no es una cadena de texto (str).
    ValueError
        Si 'dataset_origen' es una cadena vacía tras limpiar espacios.
    """
    if not isinstance(dataset_origen, str):
        raise TypeError(
            f"El parámetro 'dataset_origen' debe ser estrictamente una cadena "
            f"de texto (str). Tipo recibido: {type(dataset_origen)}."
        )

    # Sanitización de la entrada para evitar fallos por capitalización o espacios adicionales
    clean_origen = dataset_origen.strip().lower()

    if not clean_origen:
        raise ValueError(
            "El parámetro 'dataset_origen' no puede estar vacío o contener únicamente espacios."
        )

    # Bifurcación del pipeline: True si proviene del sensor real, False para datos sintéticos/externos
    return clean_origen == "zona_cero"


### **Análisis de Diseño de la Función**

1. **Robustez y Sanitización de Datos (Alineación MLOps):**  
   En un entorno de operaciones bajo presión, es común que la ingesta de metadatos o nombres de carpetas contenga errores tipográficos (ej. `"Zona_Cero"`, `"ZONA_CERO "` o `"zona_cero"`). El uso de `.strip().lower()` asegura que estas inconsistencias de entrada se resuelvan de forma determinista antes de realizar la evaluación booleana.
2. **Eficiencia de Cómputo en CPU:**  
   Al mantener esta función libre de dependencias pesadas como `librosa`, evitamos llamadas costosas al sistema operativo o procesamiento de arreglos innecesarios. Su ejecución tarda **escasas fracciones de microsegundo**, lo que permite un filtrado inmediato en el bucle principal de control antes de inicializar operaciones DSP tridimensionales.

## **Módulo: Guardado y Exportación Secuencial**

La función **`guardar_audios_preprocesados`** se ha diseñado para maximizar el rendimiento y la legibilidad en sistemas de producción que operan en la Zona Cero, se ha utilizado la librería estándar moderna **`pathlib`** junto con **`soundfile`** para una serialización binaria directa de alta velocidad, evitando cuellos de botella de hardware al descargar audios por lotes de la tarjeta SD del sensor HIKMICRO AD21P.

In [9]:
import os
from pathlib import Path
import numpy as np
import soundfile as sf
from typing import List, Union


def guardar_audios_preprocesados(
    lista_fragmentos: List[np.ndarray],
    nombre_original: str,
    dataset_origen: str,
    categoria: str,
    base_dir_salida: Union[str, Path]
) -> List[str]:
    """Guarda secuencialmente los fragmentos de audio preprocesados en disco.

    Este módulo actúa como el sumidero final de la Fase 1 del pipeline USAR V2.0.
    Establece la ruta jerárquica estricta basada en el dataset de origen y su
    categoría taxonómica, crea de forma automática las carpetas jerárquicas e
    itera secuencialmente para almacenar cada fragmento (tensor temporal de 4s)
    en formato .wav independiente, previniendo sobreescrituras incidentales.

    Parameters
    ----------
    lista_fragmentos : List[np.ndarray]
        Lista de arreglos unidimensionales de NumPy (float32) de longitud exacta 
        equivalente a 4.0 segundos a una frecuencia de muestreo de 16 kHz.
    nombre_original : str
        Nombre original de la grabación cruda de entrada (puede contener rutas,
        subrutas o extensiones como '.wav' o '.WAV').
    dataset_origen : str
        Identificador del dataset raíz (ej: "morse", "zona_cero", "voces", "esc_50").
    categoria : str
        Categoría taxonómica final de destino (ej: "Target_Rhythmic", "Target_Voice",
        "Noise_Ambient", "Noise_FalsePositive", "Noise_Structural").
    base_dir_salida : Union[str, Path]
        Ruta del directorio raíz del disco duro donde se consolidarán los datos procesados.

    Returns
    -------
    List[str]
        Lista con las rutas absolutas resueltas de todos los archivos de audio .wav
        escritos con éxito en el sistema de almacenamiento.

    Raises
    ------
    TypeError
        Si 'lista_fragmentos' no es una lista de Python o contiene elementos
        que no corresponden a arreglos estructurados de NumPy.
    ValueError
        Si la lista de fragmentos está vacía o si el parámetro 'nombre_original'
        está vacío.
    """
    # 1. Validaciones robustas de tipos de datos en la frontera de entrada (MLOps Safety)
    if not isinstance(lista_fragmentos, list):
        raise TypeError("El parámetro 'lista_fragmentos' debe ser estrictamente una lista.")

    if len(lista_fragmentos) == 0:
        raise ValueError("La lista de fragmentos de audio está vacía. No hay datos para guardar.")

    if not nombre_original:
        raise ValueError("El parámetro 'nombre_original' no puede estar vacío.")

    # 2. Construcción Dinámica de Rutas (Estructura de Carpetas USAR V2.0)
    # Ruta resultante: base_dir_salida / dataset_origen / categoria
    ruta_salida = Path(base_dir_salida) / dataset_origen / categoria
    ruta_salida.mkdir(parents=True, exist_ok=True)

    # 3. Limpieza de extensiones del nombre original utilizando Pathlib
    # stem extrae de forma limpia "rescate_01" de "C:/.../rescate_01.wav" o "/.../rescate_01.WAV"
    nombre_base = Path(nombre_original).stem

    rutas_guardadas: List[str] = []

    # 4. Exportación Secuencial Pura a 16,000 Hz
    for i, fragmento in enumerate(lista_fragmentos):
        if not isinstance(fragmento, np.ndarray):
            raise TypeError(
                f"Elemento anómalo detectado en la lista de fragmentos en el índice {i}. "
                f"Se esperaba np.ndarray, recibido: {type(fragmento)}."
            )

        # Construir nombre no colisionable concatenando un índice de parte secuencial (part0, part1...)
        nuevo_nombre_archivo = f"{nombre_base}_part{i}.wav"
        ruta_archivo_completa = ruta_salida / nuevo_nombre_archivo

        # Escribir el fragmento de audio en disco a velocidad de I/O de bajo nivel (formato FLOAT/PCM32)
        # Se fija la frecuencia de muestreo obligatoria a 16,000 Hz acordada para el pipeline
        sf.write(
            file=str(ruta_archivo_completa),
            data=fragmento,
            samplerate=16000,
            subtype='FLOAT'
        )

        # Registrar la ruta absoluta para propósitos de auditoría, debug o indexación directa
        rutas_guardadas.append(str(ruta_archivo_completa.resolve()))

    return rutas_guardadas


### **Garantías de Diseño e Integridad MLOps**

1. **Cumplimiento de la Modularidad Estricta:**  
   Esta función es **100% transparente para el cómputo de señales**. Al no incluir lógica DSP (remuestreos, filtrados o atenuación física), evitamos efectos colaterales de acoplamiento de código. Si posteriormente decidimos afinar el diseño de los filtros pasa-bajos, las rampas lineales o el cálculo de ganancia, este módulo permanecerá completamente estable e inalterado.
2. **Robustez ante Directorios Virtuales:**  
   Al emplear `Path.mkdir(parents=True, exist_ok=True)`, la clase crea de manera dinámica la jerarquía de subdirectorios completa (tanto la carpeta del dataset de origen como la subcarpeta del tipo de sonido), incluso si se ejecuta por primera vez en un ordenador limpio en la estación base, evitando excepciones por falta de directorios intermedios.
3. **Control de Ciclos y Colisiones en Disco:**  
   Cuando fraccionamos un audio de 1 minuto obtenido in situ en la Zona Cero, el bloque de segmentación temporal nos entrega una lista con múltiples fragmentos (hasta 15 partes de 4 segundos con solapamiento). Indexar cada fragmento como `{nombre_base}_part{i}.wav` asegura que cada trozo sea guardado secuencialmente en el disco duro de la estación base sin pisarse entre sí, manteniendo intacto el hilo del contexto cronológico temporal.

## **Módulo: Maestro de Preprocesamiento (MAIN)**

El **Módulo Orquestador Maestro de Preprocesamiento** de la **Fase 1** ha sido desarrollado para actuar como el **cerebro de control y enrutamiento del pipeline de datos**, integrando todas las funciones de acondicionamiento de señal, regularización acústica, lógica delta de caché y segmentación temporal continua de 4.0 segundos que hemos diseñado previamente.

In [10]:
import os
from pathlib import Path
import numpy as np
import librosa
from typing import List, Union


def orquestar_preprocesamiento_dataset(
    input_dir: Union[str, Path],
    base_dir_salida: Union[str, Path],
    dataset_origen: str,
    categoria: str,
    ruta_ruido_ambiental: Union[str, Path],
    sr_objetivo: int = 16000
) -> List[str]:
    """Orquesta el pipeline completo de preprocesamiento USAR V2.0 sobre una carpeta de audios.

    Itera sobre todos los archivos .wav en el directorio de entrada, evalúa la
    Lógica Delta (caché) para evitar el re-procesamiento redundante de audios,
    enruta condicionalmente según el hardware (Zona Cero o sintético), estandariza
    las muestras temporalmente aplicando segmentación contigua o padding por crossfade
    de 50 ms a exactamente 4.0 segundos, y finalmente escribe las señales útiles
    en disco en una jerarquía organizada de carpetas.

    Parameters
    ----------
    input_dir : Union[str, Path]
        Directorio que contiene las muestras acústicas crudas de entrada (.wav).
    base_dir_salida : Union[str, Path]
        Directorio base en disco donde se consolidarán los datos preprocesados.
    dataset_origen : str
        Identificador de procedencia del dataset (ej: "zona_cero", "morse", "voces").
    categoria : str
        Categoría taxonómica final (ej: "Target_Rhythmic", "Target_Voice").
    ruta_ruido_ambiental : Union[str, Path]
        Ruta del archivo .wav de ruido ambiental real usado como plantilla de relleno.
    sr_objetivo : int, optional
        Frecuencia de muestreo unificada del pipeline. Por defecto es 16000 Hz.

    Returns
    -------
    List[str]
        Lista ordenada con las rutas absolutas de todos los archivos de audio .wav
        que ahora residen en la carpeta de destino correspondiente en disco.

    Raises
    ------
    FileNotFoundError
        Si el directorio de entrada o la plantilla de ruido ambiental no existen.
    """
    path_input_dir = Path(input_dir)
    path_base_dir_salida = Path(base_dir_salida)
    
    # Construcción dinámica de la subcarpeta destino de producción:
    # base_dir_salida / dataset_origen / categoria
    expected_output_subfolder = path_base_dir_salida / dataset_origen / categoria
    expected_output_subfolder.mkdir(parents=True, exist_ok=True)

    # 1. CARGA DE RUIDO AMBIENTAL (Operación en Caché - Se ejecuta una sola vez)
    # Evita llamadas continuas a disco al precargar la plantilla de relleno en memoria RAM
    print(f"🔊 [ORQUESTADOR] Cargando plantilla de ruido ambiental de referencia: {ruta_ruido_ambiental}")
    try:
        ruido_ambiental, _ = librosa.load(ruta_ruido_ambiental, sr=sr_objetivo, mono=True)
        ruido_ambiental = aplicar_simulador_ad21p(ruido_ambiental, sr_objetivo, noise_std=0.0)
    except Exception as e:
        raise FileNotFoundError(
            f"❌ [ERROR] Imposible cargar la plantilla de ruido ambiental de referencia "
            f"en {ruta_ruido_ambiental}. Motivo: {e}"
        )

    if not path_input_dir.exists():
        raise FileNotFoundError(f"❌ [ERROR] El directorio de entrada de audios no existe: {input_dir}")

    # Escaneo insensible a mayúsculas/minúsculas de audios crudos en la carpeta origen
    all_wavs = [f for f in path_input_dir.iterdir() if f.is_file() and f.suffix.lower() in [".wav", ".mp3"]]
    print(f"📡 [ORQUESTADOR] Se detectaron {len(all_wavs)} archivos .wav/.mp3 en '{path_input_dir.name}'")

    # 2. BUCLE PRINCIPAL DE PROCESAMIENTO
    for file_path in all_wavs:
        filename = file_path.name
        
        nombre_original_limpio = file_path.stem
        nombre_normalizado = f"{dataset_origen}_{categoria}_{nombre_original_limpio}"

        
        # 2.1 Lógica Delta ("¿El audio es nuevo?") [Pipeline Condicional]
        # Si el audio completo o sus partes segmentadas ya existen en disco, se omiten
        if not es_audio_nuevo(nombre_normalizado, expected_output_subfolder):
            print(f"⏭️ [OMITIENDO] El archivo '{filename}' ya ha sido procesado (Caché Delta).")
            continue

        print(f"⚙️ [PROCESANDO] Iniciando pipeline de preprocesamiento para '{filename}'...")
        
        
        
        try:
            # 2.2 Carga Acústica Base (Remuestreo forzado a mono y 16,000 Hz)
            audio, _ = librosa.load(file_path, sr=sr_objetivo, mono=True)
            
            # 2.3 Condicional de Hardware ("¿El audio es de Dataset Zona Cero?")
            is_real_hardware = es_dataset_zona_cero(dataset_origen)
            
            if not is_real_hardware:
                # Si proviene de datasets externos o sintéticos, simular respuesta física
                # del HIKMICRO AD21p (pasa-bajos de oclusión)
                audio = aplicar_simulador_ad21p(audio, sr_objetivo, noise_std=0.0)
                
            # 2.3.1 Data Augmentation (Exclusivo para la clase Rítmica/Morse)
            if es_dataset_morse(dataset_origen):
                audio = simular_variabilidad_impacto(
                    audio=audio, 
                    sr=sr_objetivo, 
                    variar_cadencia=True, 
                    variar_material=True
                )
                
            # 2.4 Condicional Temporal ("¿El audio es mayor de 4s?")
            duracion = len(audio) / sr_objetivo
            lista_fragmentos: List[np.ndarray] = []

            if duracion < 4.0:
                # Audio corto: Centralización geométrica y padding mediante crossfade continuo
                fragmento_con_padding = padding_por_crossfade(
                    senal_util=audio,
                    ruido_ambiental=ruido_ambiental,
                    target_duration=4.0,
                    sr=sr_objetivo
                )
                lista_fragmentos.append(fragmento_con_padding)
            else:
                # Audio largo: Desglose contiguo en ventanas de 4s y procesamiento del residuo
                lista_fragmentos = segmentar_y_estandarizar_audio(
                    senal_util=audio,
                    ruido_ambiental=ruido_ambiental,
                    sr=sr_objetivo,
                    target_duration=4.0,
                    min_duration_residual=1.0
                )


            # 2.5 Inyección de Estática Electrónica (AWGN Global Post-Ensamblaje)
            lista_fragmentos_finales: List[np.ndarray] = []
            if lista_fragmentos:
                for fragmento in lista_fragmentos:
                    if not is_real_hardware:
                        # La estática se inyecta a todo el tensor de 4.0s (voz + ruido ambiental)
                        fragmento = inyectar_ruido_estatico(fragmento)
                    lista_fragmentos_finales.append(fragmento)

                    
            # 2.5 Guardado Estructurado
            if lista_fragmentos_finales:
                guardados = guardar_audios_preprocesados(
                    lista_fragmentos=lista_fragmentos_finales,
                    nombre_original=nombre_normalizado,
                    dataset_origen=dataset_origen,
                    categoria=categoria,
                    base_dir_salida=base_dir_salida
                )
                print(f"✅ [ÉXITO] '{filename}' preprocesado con éxito en {len(guardados)} partes de 4s.")
            else:
                print(f"⚠️ [AVISO] '{filename}' descartado por poseer duración insuficiente (residuo < 1.0s).")

        except Exception as e:
            # Salvaguarda MLOps: si un .wav está corrupto, trunca, o daña los punteros en disco,
            # se loguea la excepción pero el bucle de producción continúa analizando el resto del dataset.
            print(f"❌ [ERROR] Fallo en el pipeline para '{filename}': {e}")
            continue

    # 3. CONSOLIDACIÓN DE RUTAS FINALES DE PRODUCCIÓN
    rutas_finales = []
    if expected_output_subfolder.exists():
        rutas_finales = [
            str(f.resolve()) for f in expected_output_subfolder.iterdir() 
            if f.is_file() and f.suffix.lower() == ".wav"
        ]

    return sorted(rutas_finales)


### **Garantías de Diseño e Integridad en el Terreno**

1. **Alineación Exacta con el Diagrama de Flujo (Draw.io):**  
   Este orquestador es la **traducción matemática directa** de los bloques condicionales e iteradores que tu comité académico y los ingenieros de rescate validaron visualmente en el plano de Draw.io [Pipeline_Deteccion_Vida.drawio (1).pdf]. Cada decisión (*¿Es Zona Cero?*, *¿Es mayor de 4 segundos?*) está acoplada al código en el orden de procedencia adecuado.
2. **Robustez Antifallos en la Zona Cero (Fault-Tolerant Loop):**  
   Durante operaciones reales bajo escombros, las laptops tácticas de campo pueden sufrir cortes de energía o bloqueos. Integrar el bloque `try/except` a nivel de archivo individual garantiza que si el pipeline se topa con una muestra truncada o con bytes corruptos de una tarjeta SD dañada, **el script no se detendrá**; registrará la alerta y continuará procesando el resto del bloque de audios.
3. **Eficiencia en el Cómputo del Ruido:**  
   En lugar de abrir e ingestar en disco el archivo de ruido ambiental por cada fragmento, la plantilla de 4.0 segundos de referencia se almacena de forma única en caché de memoria RAM al iniciar el script, reduciendo drásticamente las operaciones I/O innecesarias y el estrés del procesador i9.
4. **Lógica Delta Integrada:**  
   Al interactuar con `es_audio_nuevo`, el orquestador comprueba si una muestra ya fue dividida y guardada. Esto permite ejecutar este pipeline de forma recurrente (*Active Learning loop*) de manera que solo se procesen los nuevos archivos que el transductor capture día a día, ahorrando ciclos de cómputo inestimables en el terreno [SISTEMA USAR V2.0.txt].

# **EJECUCION FASE 1**

In [ ]:
# 1. Definir las rutas y variables reales de tu entorno
ruta_entrada = r"C:\Users\carlo\Documents\detector_vida_acustico\detector_vida_acustico\dataset_original\zona_cero\Target_Rhythmic" 
ruta_salida_base = r"C:\Users\carlo\Documents\detector_vida_acustico\detector_vida_acustico\dataset_preprocesado"
dataset_actual = "zona_cero"
categoria_actual = "Target_Rhythmic"
ruta_ruido = r"C:\Users\carlo\Documents\detector_vida_acustico\detector_vida_acustico\dataset_original\zona_cero\Noise_Ambient\base_ambiente_referencia_padding.wav"

# 2. Ejecutar el orquestador pasando las variables
rutas_procesadas = orquestar_preprocesamiento_dataset(
    input_dir=ruta_entrada,
    base_dir_salida=ruta_salida_base,
    dataset_origen=dataset_actual,
    categoria=categoria_actual,
    ruta_ruido_ambiental=ruta_ruido,
    sr_objetivo=16000 # Opcional, ya está por defecto
)

print(f"\n🚀 Fase 1 Completada. Total de tensores generados: {len(rutas_procesadas)}")

In [11]:
import os
from pathlib import Path

def ejecutar_pipeline_batch():
    print("======================================================")
    print(" INICIANDO PIPELINE DE INGESTA USAR V2.0 (BATCH MODE) ")
    print("======================================================\n")

    # 1. Definición de Rutas Base (RUTAS ABSOLUTAS CORREGIDAS)
    ruta_base_original = Path(r"C:\Users\carlo\Documents\detector_vida_acustico\detector_vida_acustico\dataset_original")
    ruta_base_preprocesado = Path(r"C:\Users\carlo\Documents\detector_vida_acustico\detector_vida_acustico\dataset_preprocesado")
    
    # Ruta absoluta de la plantilla de ruido ambiental
    ruta_ruido_ambiental = ruta_base_original / "zona_cero" / "Noise_General" / "base_ambiente_referencia_padding.wav"

    # 2. Definición de Topología de Datos
    datasets = [
        'esr_50', 
        'morse', 
        'sonidos_perros_gatos', 
        'urban_sound', 
        'voces_espanol', 
        'zona_cero',
        'gritos'
    ]
    
    categorias = [
        "Noise_General", 
        "Target_Animal", 
        "Target_Rhythmic", 
        "Target_Voice"
    ]

    total_audios_procesados = 0

    # 3. Bucle de Enrutamiento Automático
    for dataset in datasets:
        for categoria in categorias:
            input_dir = ruta_base_original / dataset / categoria
            
            # Verificar si la combinación dataset/categoría existe en disco
            if not input_dir.exists():
                continue  # Salta esta iteración si la carpeta no existe
                
            print(f"\n🚀 Iniciando Lote -> Dataset: [{dataset.upper()}] | Categoría: [{categoria}]")
            
            # Llamada al orquestador por cada subcarpeta encontrada
            rutas_generadas = orquestar_preprocesamiento_dataset(
                input_dir=input_dir,
                base_dir_salida=ruta_base_preprocesado,
                dataset_origen=dataset,
                categoria=categoria,
                ruta_ruido_ambiental=ruta_ruido_ambiental,
                sr_objetivo=16000
            )
            
            total_audios_procesados += len(rutas_generadas)
            print(f"✅ Lote completado. Tensores generados: {len(rutas_generadas)}")

    print("\n======================================================")
    print(f" PIPELINE FINALIZADO. Total de fragmentos 4s generados: {total_audios_procesados}")
    print("======================================================")

# Ejecutar el proceso masivo
ejecutar_pipeline_batch()

 INICIANDO PIPELINE DE INGESTA USAR V2.0 (BATCH MODE) 


🚀 Iniciando Lote -> Dataset: [SONIDOS_PERROS_GATOS] | Categoría: [Target_Animal]
🔊 [ORQUESTADOR] Cargando plantilla de ruido ambiental de referencia: C:\Users\carlo\Documents\detector_vida_acustico\detector_vida_acustico\dataset_original\zona_cero\Noise_General\base_ambiente_referencia_padding.wav
📡 [ORQUESTADOR] Se detectaron 277 archivos .wav/.mp3 en 'Target_Animal'
⚙️ [PROCESANDO] Iniciando pipeline de preprocesamiento para 'cat_1.wav'...
✅ [ÉXITO] 'cat_1.wav' preprocesado con éxito en 3 partes de 4s.
⚙️ [PROCESANDO] Iniciando pipeline de preprocesamiento para 'cat_10.wav'...
✅ [ÉXITO] 'cat_10.wav' preprocesado con éxito en 3 partes de 4s.
⚙️ [PROCESANDO] Iniciando pipeline de preprocesamiento para 'cat_100.wav'...
✅ [ÉXITO] 'cat_100.wav' preprocesado con éxito en 1 partes de 4s.
⚙️ [PROCESANDO] Iniciando pipeline de preprocesamiento para 'cat_101.wav'...
✅ [ÉXITO] 'cat_101.wav' preprocesado con éxito en 3 partes de 4s.
⚙️ [PR